In [2]:
import pandas as pd
import numpy as np

In [4]:
sample_submission = pd.read_csv("../data/SampleSubmissionStage1.csv")
team_features_men = pd.read_csv("../data/m_team_season_features.csv")
train_df = pd.read_csv("../data/m_tournament_training_dataset_advanced.csv")

In [5]:
submission_matchups = sample_submission.copy()

submission_matchups[["Season", "Team1ID", "Team2ID"]] = (
    submission_matchups["ID"]
    .str.split("_", expand=True)
)

submission_matchups["Season"] = submission_matchups["Season"].astype(int)
submission_matchups["Team1ID"] = submission_matchups["Team1ID"].astype(int)
submission_matchups["Team2ID"] = submission_matchups["Team2ID"].astype(int)


In [6]:
team1_features = team_features_men.add_prefix("Team1_")
team2_features = team_features_men.add_prefix("Team2_")

submission_matchups = submission_matchups.merge(
    team1_features,
    left_on=["Season", "Team1ID"],
    right_on=["Team1_Season", "Team1_TeamID"],
    how="left"
)

In [7]:
submission_matchups = submission_matchups.merge(
    team2_features,
    left_on=["Season", "Team2ID"],
    right_on=["Team2_Season", "Team2_TeamID"],
    how="left"
)

print("Submission matchup shape after merges:", submission_matchups.shape)
print(submission_matchups.head())

Submission matchup shape after merges: (519144, 97)
               ID  Pred  Season  Team1ID  Team2ID  Team1_Season  Team1_TeamID  \
0  2022_1101_1102   0.5    2022     1101     1102        2022.0        1101.0   
1  2022_1101_1103   0.5    2022     1101     1103        2022.0        1101.0   
2  2022_1101_1104   0.5    2022     1101     1104        2022.0        1101.0   
3  2022_1101_1105   0.5    2022     1101     1105        2022.0        1101.0   
4  2022_1101_1106   0.5    2022     1101     1106        2022.0        1101.0   

   Team1_Wins  Team1_GamesPlayed  Team1_AvgPointsFor  ...  Team2_AvgNetRating  \
0        19.0               29.0           73.172414  ...           -0.115917   
1        19.0               29.0           73.172414  ...            0.080546   
2        19.0               29.0           73.172414  ...            0.048862   
3        19.0               29.0           73.172414  ...           -0.073582   
4        19.0               29.0           73.172414  ..

In [8]:
submission_matchups["SeedNumDiff"] = (
        submission_matchups["Team1_SeedNum"] - submission_matchups["Team2_SeedNum"]
)

submission_matchups["RankingDiff"] = (
        submission_matchups["Team1_MasseyOrdinalRank"] - submission_matchups["Team2_MasseyOrdinalRank"]
)

submission_matchups["MarginDiff"] = (
        submission_matchups["Team1_AvgMarginScore"] - submission_matchups["Team2_AvgMarginScore"]
)

submission_matchups["NetRatingDiff"] = (
        submission_matchups["Team1_AvgNetRating"] - submission_matchups["Team2_AvgNetRating"]
)

submission_matchups["OffEffDiff"] = (
        submission_matchups["Team1_AvgOffEfficiency"] - submission_matchups["Team2_AvgOffEfficiency"]
)

submission_matchups["DefEffDiff"] = (
        submission_matchups["Team1_AvgDefEfficiency"] - submission_matchups["Team2_AvgDefEfficiency"]
)

submission_matchups["WinPctDiff"] = (
        submission_matchups["Team1_WinPct"] - submission_matchups["Team2_WinPct"]
)

submission_matchups["TurnoverMarginDiff"] = (
        submission_matchups["Team1_AvgTurnoverMargin"] - submission_matchups["Team2_AvgTurnoverMargin"]
)

submission_matchups["ReboundPctDiff"] = (
        submission_matchups["Team1_AvgReboundPct"] - submission_matchups["Team2_AvgReboundPct"]
)

# Engineered features from your training logic
submission_matchups["OffDefGap"] = (
        submission_matchups["OffEffDiff"] - submission_matchups["DefEffDiff"]
)

submission_matchups["DominanceScore"] = (
        submission_matchups["MarginDiff"] * submission_matchups["WinPctDiff"]
)

submission_matchups["NetRating_Margin_Interaction"] = (
        submission_matchups["NetRatingDiff"] * submission_matchups["MarginDiff"]
)

submission_matchups["Margin_Ranking_Interaction"] = (
        submission_matchups["MarginDiff"] * submission_matchups["RankingDiff"]
)


C:\Users\Owner\AppData\Local\Temp\ipykernel_37964\2810408252.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  submission_matchups["ReboundPctDiff"] = (
C:\Users\Owner\AppData\Local\Temp\ipykernel_37964\2810408252.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  submission_matchups["OffDefGap"] = (
C:\Users\Owner\AppData\Local\Temp\ipykernel_37964\2810408252.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joi

In [9]:
model_features = [
    'SeedNumDiff',
    'RankingDiff',
    'MarginDiff',
    'NetRatingDiff',
    'OffEffDiff',
    'DefEffDiff',
    'WinPctDiff',
    'OffDefGap',
    'DominanceScore',
    'NetRating_Margin_Interaction',
    'Margin_Ranking_Interaction',
    'TurnoverMarginDiff',
    'ReboundPctDiff'
]

In [10]:
X_submit = submission_matchups[model_features].copy()

In [11]:
print("X_submit shape:", X_submit.shape)
print(X_submit.head())

X_submit shape: (519144, 13)
   SeedNumDiff  RankingDiff  MarginDiff  NetRatingDiff  OffEffDiff  \
0          NaN       -104.0   11.758621       0.177664    0.067473   
1          NaN         12.0   -0.370412      -0.018799   -0.063657   
2          NaN        105.0    1.196121       0.012886   -0.062496   
3          NaN       -194.0   10.325287       0.135329    0.136502   
4          NaN       -203.0   11.358621       0.152044    0.061059   

   DefEffDiff  WinPctDiff  OffDefGap  DominanceScore  \
0   -0.110191    0.275862   0.177664        3.243757   
1   -0.044858   -0.054505  -0.018799        0.020189   
2   -0.075382    0.061422   0.012886        0.073469   
3    0.001173    0.255172   0.135329        2.634728   
4   -0.090984    0.355172   0.152044        4.034269   

   NetRating_Margin_Interaction  Margin_Ranking_Interaction  \
0                      2.089082                -1222.896552   
1                      0.006963                   -4.444939   
2                      0

In [12]:
season_cutoff = 2023
train_df_filtered = train_df[train_df["Season"] < season_cutoff].copy()

X_train = train_df_filtered[model_features].copy()
y_train = train_df_filtered["Target"].copy()

In [13]:
from sklearn.ensemble import RandomForestClassifier

final_rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=5,
    min_samples_split=15,
    random_state=42,
    n_jobs=-1
)

In [14]:
X_train = train_df[model_features].copy()
y_train = train_df["Target"].copy()

final_rf_model.fit(X_train, y_train)

print("Final model trained.")
print("Training shape:", X_train.shape)

Final model trained.
Training shape: (2898, 13)


In [15]:
submission_matchups["Pred"] = final_rf_model.predict_proba(X_submit)[:, 1]

In [16]:
final_submission = submission_matchups[["ID", "Pred"]].copy()

In [ ]:
final_submission.to_csv("../submissions/submission_stage1.csv", index=False)

In [17]:
print(final_submission.head())
print("Submission file saved.")

               ID      Pred
0  2022_1101_1102  0.850208
1  2022_1101_1103  0.474597
2  2022_1101_1104  0.376967
3  2022_1101_1105  0.842144
4  2022_1101_1106  0.832098
Submission file saved.


In [18]:
print(X_submit.isna().sum())

SeedNumDiff                     510032
RankingDiff                     258131
MarginDiff                      258131
NetRatingDiff                   258131
OffEffDiff                      258131
DefEffDiff                      258131
WinPctDiff                      258131
OffDefGap                       258131
DominanceScore                  258131
NetRating_Margin_Interaction    258131
Margin_Ranking_Interaction      258131
TurnoverMarginDiff              258131
ReboundPctDiff                  258131
dtype: int64
